In [ ]:
##REPASO DE SQL BASICO CON BASE DE DATOS SOCCER

# PRIMERO INSPECCIONAMOS LA BASE DE DATOS PARA VER SU ESTRUCTURA 

from sqlalchemy import create_engine, inspect

# Define el nombre de la base de datos SQLite
db_name = 'database.sqlite'
engine = create_engine(f'sqlite:///{db_name}')

# Crea un objeto inspector para examinar la base de datos
inspector = inspect(engine)

# Obtiene la lista de todos los nombres de las tablas
table_names = inspector.get_table_names()

print('Esquema de la base de datos:')
for table_name in table_names:
    print(f'\nTabla: {table_name}')
    columns = inspector.get_columns(table_name)
    print('  Columnas:')
    for column in columns:
        print(f'    - {column["name"]} ({column["type"]}), Primary Key: {column["primary_key"]}, Nullable: {column["nullable"]}')
    foreign_keys = inspector.get_foreign_keys(table_name)
    if foreign_keys:
        print('  Claves Foráneas:')
        for fk in foreign_keys:
            referenced_table = fk["referred_table"]
            local_columns = fk["constrained_columns"]
            referenced_columns = fk["referred_columns"]
            print(f'    - {local_columns} referencia a {referenced_table}.{referenced_columns}')

# Cierra la conexión a la base de datos
engine.dispose()

Esquema de la base de datos:

Tabla: Country
  Columnas:
    - id (INTEGER), Primary Key: 1, Nullable: True
    - name (TEXT), Primary Key: 0, Nullable: True

Tabla: League
  Columnas:
    - id (INTEGER), Primary Key: 1, Nullable: True
    - country_id (INTEGER), Primary Key: 0, Nullable: True
    - name (TEXT), Primary Key: 0, Nullable: True
  Claves Foráneas:
    - ['country_id'] referencia a country.['id']

Tabla: Match
  Columnas:
    - id (INTEGER), Primary Key: 1, Nullable: True
    - country_id (INTEGER), Primary Key: 0, Nullable: True
    - league_id (INTEGER), Primary Key: 0, Nullable: True
    - season (TEXT), Primary Key: 0, Nullable: True
    - stage (INTEGER), Primary Key: 0, Nullable: True
    - date (TEXT), Primary Key: 0, Nullable: True
    - match_api_id (INTEGER), Primary Key: 0, Nullable: True
    - home_team_api_id (INTEGER), Primary Key: 0, Nullable: True
    - away_team_api_id (INTEGER), Primary Key: 0, Nullable: True
    - home_team_goal (INTEGER), Primary Key: 0

Clase de Repaso: Consolidando SQL con Datos de Fútbol (Énfasis en JOINs)
Duración: 1 hora y 45 minutos (aproximadamente)

Objetivos de Aprendizaje:
Al final de esta clase, el estudiante será capaz de:

Reforzar el uso de SELECT, FROM, WHERE, GROUP BY, HAVING, ORDER BY y funciones agregadas.
Dominar la aplicación de INNER JOIN para combinar datos relevantes de múltiples tablas.
Comprender y aplicar LEFT JOIN para incluir todas las filas de una tabla principal, incluso sin coincidencias.
Analizar y resolver problemas de datos complejos que requieren uniones de múltiples tablas.
Sección 1: Repaso Rápido de Conceptos Clave (15 minutos)
¿Por qué normalizamos los datos? Evitar redundancia, mejorar integridad.
Claves Primarias (PK) y Claves Foráneas (FK): La espina dorsal de los JOINs.
Ejemplos en la DB de fútbol: Country.id es PK, League.country_id es FK referenciando Country.id. 
Team.team_api_id es PK, Match.home_team_api_id y Match.away_team_api_id son FKs referenciando a Team.team_api_id. 


Player.player_api_id es PK, Match.home_player_1 a home_player_11 y away_player_1 a away_player_11 son FKs. 

Repaso de cláusulas (orden de ejecución): FROM -> JOIN -> WHERE -> GROUP BY -> HAVING -> SELECT -> ORDER BY -> LIMIT.
Sección 2: Profundizando en INNER JOIN con Ejemplos (45 minutos)
El INNER JOIN es el más común y útil para combinar filas donde hay correspondencia en ambas tablas.

Concepto: Solo devuelve las filas donde hay una coincidencia en ambas tablas.

Ejemplo 1: Nombres de Ligas y Sus Países

Pregunta: ¿Cómo obtenemos el nombre de cada liga junto con el nombre del país al que pertenece?
Explicación: La tabla League tiene country_id que es un FK a Country.id. Necesitamos unir estas dos tablas.

In [ ]:
import pandas as pd
query = """
SELECT
    L.name AS league_name,
    C.name AS country_name
FROM
    League AS L
INNER JOIN
    Country AS C 
ON L.country_id = C.id
LIMIT 10;
"""
df = pd.read_sql_query(query, engine)

In [3]:
df

,league_name,country_name
0,Belgium Jupiler League,Belgium
1,England Premier League,England
2,France Ligue 1,France
3,Germany 1. Bundesliga,Germany
4,Italy Serie A,Italy
5,Netherlands Eredivisie,Netherlands
6,Poland Ekstraklasa,Poland
7,Portugal Liga ZON Sagres,Portugal
8,Scotland Premier League,Scotland
9,Spain LIGA BBVA,Spain


Ejemplo 2: Partidos con Nombres de Equipos y Liga

Pregunta: ¿Cómo mostrar el nombre de los equipos (local y visitante) y el nombre de la liga para un conjunto de partidos?
Explicación: La tabla Match tiene home_team_api_id y away_team_api_id que son FKs a Team.team_api_id. También tiene league_id que es FK a League.id. Necesitaremos tres INNER JOINs.

In [ ]:
#VEAMOS LOS NOMBRES DE LAS LIGAS
query = """ 
SELECT * FROM League
"""
df = pd.read_sql_query(query, engine)

In [10]:
df

,id,country_id,name
0,1,1,Belgium Jupiler League
1,1729,1729,England Premier League
2,4769,4769,France Ligue 1
3,7809,7809,Germany 1. Bundesliga
4,10257,10257,Italy Serie A
5,13274,13274,Netherlands Eredivisie
6,15722,15722,Poland Ekstraklasa
7,17642,17642,Portugal Liga ZON Sagres
8,19694,19694,Scotland Premier League
9,21518,21518,Spain LIGA BBVA


In [12]:
query = """
SELECT
    M.date,
    T_home.team_long_name AS home_team,
    T_away.team_long_name AS away_team,
    M.home_team_goal,
    M.away_team_goal,
    L.name AS league_name
FROM
    Match AS M
INNER JOIN
    Team AS T_home ON M.home_team_api_id = T_home.team_api_id
INNER JOIN
    Team AS T_away ON M.away_team_api_id = T_away.team_api_id
INNER JOIN
    League AS L ON M.league_id = L.id
WHERE
    M.season = '2015/2016' AND L.name = 'Spain LIGA BBVA'
LIMIT 5;
"""
df = pd.read_sql_query(query, engine)

In [13]:
df

,date,home_team,away_team,home_team_goal,away_team_goal,league_name
0,2015-08-23 00:00:00,Levante UD,RC Celta de Vigo,1,2,Spain LIGA BBVA
1,2015-08-22 00:00:00,Atlético Madrid,UD Las Palmas,1,0,Spain LIGA BBVA
2,2015-08-21 00:00:00,Málaga CF,Sevilla FC,0,0,Spain LIGA BBVA
3,2015-08-23 00:00:00,Athletic Club de Bilbao,FC Barcelona,0,1,Spain LIGA BBVA
4,2015-08-24 00:00:00,Granada CF,SD Eibar,1,3,Spain LIGA BBVA


Sección 3: Profundizando en LEFT JOIN con Ejemplos (30 minutos)
El LEFT JOIN es crucial cuando queremos mantener todas las filas de la tabla "izquierda", incluso si no hay coincidencias en la "derecha".

Concepto: Devuelve todas las filas de la tabla izquierda y las filas coincidentes de la tabla derecha. Si no hay coincidencia, las columnas de la tabla derecha son NULL.

Ejemplo 3: Países y Sus Ligas (Incluso si un País no tiene Liga Registrada)

Pregunta: ¿Cómo listar todos los países y, si tienen, el nombre de las ligas asociadas? Si un país no tiene ligas registradas en la DB, aún debe aparecer.
Explicación: Queremos todos los países, así que Country es nuestra tabla "izquierda". Unimos con League.

In [14]:
query = """
SELECT
    C.name AS country_name,
    L.name AS league_name
FROM
    Country AS C
LEFT JOIN
    League AS L ON C.id = L.country_id
ORDER BY
    C.name, L.name
LIMIT 10;
"""
df = pd.read_sql_query(query, engine)

In [15]:
df

,country_name,league_name
0,Belgium,Belgium Jupiler League
1,England,England Premier League
2,France,France Ligue 1
3,Germany,Germany 1. Bundesliga
4,Italy,Italy Serie A
5,Netherlands,Netherlands Eredivisie
6,Poland,Poland Ekstraklasa
7,Portugal,Portugal Liga ZON Sagres
8,Scotland,Scotland Premier League
9,Spain,Spain LIGA BBVA


Ejemplo 4: Equipos y el Número de Partidos Como Visitante sin Goles

Pregunta: ¿Cómo listar todos los equipos (Team) y, para cada uno, cuántos partidos jugaron como equipo visitante en los que no anotaron goles (away_team_goal = 0)? Los equipos que nunca jugaron como visitante o que siempre anotaron deben aparecer con conteo 0.
Explicación: Team es la tabla izquierda. Unimos con Match donde el equipo es el visitante y los goles son 0. Usamos COUNT con GROUP BY.

In [16]:
query = """
SELECT
    T.team_long_name,
    COUNT(M.id) AS matches_without_away_goal
FROM
    Team AS T
LEFT JOIN
    Match AS M ON T.team_api_id = M.away_team_api_id AND M.away_team_goal = 0
GROUP BY
    T.team_long_name
ORDER BY
    matches_without_away_goal DESC
LIMIT 10;
"""
df = pd.read_sql_query(query, engine)

In [17]:
df

,team_long_name,matches_without_away_goal
0,RCD Espanyol,69
1,Stoke City,68
2,Polonia Bytom,66
3,OGC Nice,66
4,Stade Rennais FC,63
5,Getafe CF,62
6,Sunderland,60
7,Genoa,60
8,Chievo Verona,60
9,Toulouse FC,59


Ejercicios de SQL: Refuerzo de JOINs y Agregación con Datos de Fútbol

1. Ejercicios de INNER JOIN

A. Goles Totales por País en la Temporada 2015/2016:

Pregunta: ¿Cuál es la suma total de goles (locales + visitantes) anotados en la temporada '2015/2016' para cada país?
Muestra: El nombre del país y la suma de goles.
Ordena: De mayor a menor número de goles.

B. Partidos de la Spain LIGA BBVA en 2014/2015:

Pregunta: ¿Cuáles son la fecha (date), el equipo local (home_team_long_name), el equipo visitante (away_team_long_name) y el resultado de goles (home_team_goal, away_team_goal) para todos los partidos de la liga 'Spain La Liga' en la temporada '2014/2015'?
Ordena: Por fecha ascendente.

C. Jugadores con más de 100 Goles como Anotadores (Si es posible):

Nota: La columna goal en la tabla Match es TEXT. Si contiene JSON o texto que lista anotadores, este ejercicio requeriría análisis de texto avanzado fuera del alcance de SQL puro o funciones específicas del SGBD que SQLite no tiene fácilmente.
Pregunta (Simplificada): ¿Cuántos partidos tuvieron goles anotados por el equipo local y el equipo visitante en la temporada '2015/2016' en la 'England Premier League'? (Solo cuenta el match_id si ambos anotaron).

2. Ejercicios de LEFT JOIN

A. Países sin Partidos Registrados en 2016/2017:

Pregunta: ¿Cuáles son los nombres de los países que no tienen ningún partido registrado en la temporada '2016/2017'?
Muestra: Solo el nombre del país.

B. Equipos y el Número de Partidos Jugados como Local (incluyendo 0):

Pregunta: ¿Cuántos partidos ha jugado cada equipo como equipo local? Incluye a los equipos que no hayan jugado ningún partido como local.
Muestra: El nombre largo del equipo y el conteo de partidos.
Ordena: Por el conteo de partidos de forma descendente.
Ejercicios Combinados y Complejos

C. Ligas con Promedio de Goles Superior a 2.5 por Partido:

Pregunta: ¿Cuáles son los nombres de las ligas (y sus respectivos países) donde el promedio de goles totales por partido (home_team_goal + away_team_goal) es superior a 2.5 en la temporada '2015/2016'?
Muestra: Nombre de la liga, nombre del país y el promedio de goles.
Ordena: Por el promedio de goles de forma descendente.

D. Equipos que Nunca Perdieron como Local en la Temporada 2013/2014:

Pregunta: ¿Cuáles son los nombres largos de los equipos que no perdieron ningún partido como equipo local en la temporada '2013/2014'?
Muestra: Solo el nombre largo del equipo.

E. Jugador Más Alto y Más Pesado por País de Liga:

Pregunta: ¿Cuál es el nombre del jugador más alto y el jugador más pesado que ha jugado en partidos de cada país (de las ligas)?
Muestra: El nombre del país, el nombre del jugador más alto con su altura, y el nombre del jugador más pesado con su peso. (Esto es muy complejo si se busca el jugador en sí, simplificar a MAX(height) y MAX(weight) por país es más realista para este nivel, si no, se requieren subconsultas correlacionadas o CTEs avanzadas).
Simplificado: ¿Cuál es la altura máxima y el peso máximo de los jugadores que han participado en partidos de cada país (de la liga)?
Muestra: Nombre del país, altura máxima y peso máximo.

F. Partidos donde Ambos Equipos son del Mismo País y la Liga es Conocida:

Pregunta: ¿Cuántos partidos hay donde el equipo local y el equipo visitante pertenecen al mismo país, y ese país también está asociado a una liga?
Muestra: El nombre del país y el conteo de partidos.
Ordena: Por el conteo de partidos descendente.

G. Ligas con Más de 2000 Goles Totales en un Año Específico:

Pregunta: ¿Cuáles son los nombres de las ligas y los años donde el total de goles anotados (home_team_goal + away_team_goal) supera los 2000?
Muestra: Nombre de la liga, el año y el total de goles.
Ordena: Por el total de goles descendente.